In [266]:
import pdfplumber
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [267]:
pdf = pdfplumber.open("divipole_geolocations.pdf")

In [268]:
first_page = pdf.pages[0]
table = first_page.extract_table()
divipole = pd.DataFrame(table[6:], columns=table[5])
columns = divipole.columns

In [269]:
drop_columns = [*columns[15:], "Mujeres", "Hombres", "Comuna", "Dirección"]
index_drop_columns = [divipole.columns.get_loc(col) for col in drop_columns]

In [270]:
divipole.drop(columns=drop_columns, inplace=True)
keep_columns = divipole.columns

In [271]:
progress = tqdm(pdf.pages[1:], total=len(pdf.pages) - 1, desc="Extracting tables from PDF pages")
for page in progress:
    table = page.extract_table()
    df = pd.DataFrame(table)
    df.drop(columns=index_drop_columns, inplace=True)
    df.columns = keep_columns
    divipole = pd.concat([divipole, df], ignore_index=True)

Extracting tables from PDF pages:   0%|          | 0/191 [00:00<?, ?it/s]

In [272]:
divipole_loc = divipole[divipole["Latitud"].notna() & divipole["Longitud"].notna()]

In [278]:
divipole_loc.to_csv("divipole_geolocations.csv", index=False)